# 📗 Module 04 · Notebook 07 — Deploy SLM Spesialis ke Edge (Mac + Jetson)

Notebook 05/06 **melatih** spesialis. Sekarang kita **kirim ke lapangan**: ubah adapter QLoRA jadi
**GGUF**, push ke HuggingFace, lalu jalankan di **Mac** & **Jetson Orin Nano 8GB**.

## Alur (capstone)
`adapter QLoRA → merge fp16 → GGUF → quantize → push HF (chmdznr) → Ollama (demo di Colab) → device asli (PDF runbook)`

## Target hardware
- **MacBook** (Apple Silicon) — dev lokal
- **Jetson Orin Nano 8GB** — JetPack 6.2.1, CUDA 12.6, Ollama 0.30.6 sudah terpasang

> Notebook ini membuktikan pipeline + demo Ollama hidup di Colab. Langkah di device asli ada di
> **PDF Runbook** terpisah.

In [ ]:
!pip install -q "transformers>=4.53,<5" peft accelerate bitsandbytes datasets huggingface_hub openai
!nvidia-smi -L

## 1. Siapkan Adapter Spesialis

Kita latih ulang adapter ekstraksi-JSON singkat (resep notebook 05, ~2-3 menit) agar notebook
*self-contained*. Hasilnya: `./qlora-adapter`.

In [ ]:
import torch, json, random, gc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import DataCollatorForLanguageModeling, Trainer, TrainingArguments

BASE = "Qwen/Qwen3-0.6B"   # non-gated, mungil
SYS  = "Ekstrak pesanan ke JSON dengan key: produk, jumlah, harga_satuan, total. Angka tanpa titik. Keluarkan HANYA JSON."

random.seed(0)
PRODUK=["beras","minyak goreng","gula","kopi","sabun","mie instan","teh","garam"]
def gen_one():
    p=random.choice(PRODUK); j=random.randint(1,20); h=random.choice([5000,12000,14000,25000,30000])
    text=f"Pesan {j} {p}, harga {h:,} per item.".replace(",",".")
    return {"text":text,"gold":{"produk":p,"jumlah":j,"harga_satuan":h,"total":j*h}}
train_data=[gen_one() for _ in range(120)]

tok=AutoTokenizer.from_pretrained(BASE)
if tok.pad_token is None: tok.pad_token=tok.eos_token
bnb=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                       bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
model=AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, device_map="auto")

def fmt(ex):
    msgs=[{"role":"system","content":SYS},{"role":"user","content":ex["text"]},
          {"role":"assistant","content":json.dumps(ex["gold"], ensure_ascii=False)}]
    try: return {"t": tok.apply_chat_template(msgs, tokenize=False, enable_thinking=False)}
    except TypeError: return {"t": tok.apply_chat_template(msgs, tokenize=False)}
ds=Dataset.from_list(train_data).map(fmt)
ds=ds.map(lambda e: tok(e["t"], truncation=True, max_length=200), remove_columns=ds.column_names)

model=prepare_model_for_kbit_training(model)
model=get_peft_model(model, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM", target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]))
Trainer(model=model, args=TrainingArguments(output_dir="t", per_device_train_batch_size=8,
    num_train_epochs=6, learning_rate=2e-4, fp16=True, logging_steps=20, optim="paged_adamw_8bit",
    report_to="none", save_strategy="no"),
    train_dataset=ds, data_collator=DataCollatorForLanguageModeling(tokenizer=tok, mlm=False)).train()
model.save_pretrained("./qlora-adapter"); tok.save_pretrained("./qlora-adapter")
print("adapter -> ./qlora-adapter")
del model; gc.collect(); torch.cuda.empty_cache()

## 2. ⚠️ Jebakan #1 — JANGAN merge ke base 4-bit

Kalau kamu `merge_and_unload()` di atas base yang masih 4-bit, NaN tersuntik saat konversi → GGUF
menghasilkan **output `GGGG…`** (bug nyata: Unsloth #2860). **Aturan emas:** reload base
**fp16**, baru attach adapter & merge.

In [ ]:
from peft import PeftModel
OUT="./qwen3-merged-fp16"
# KRITIS: fp16, BUKAN 4-bit. device_map="cpu" aman dari OOM (merge tak butuh GPU).
base=AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float16, device_map="cpu")
merged=PeftModel.from_pretrained(base, "./qlora-adapter").merge_and_unload()
merged.save_pretrained(OUT, safe_serialization=True)
AutoTokenizer.from_pretrained(BASE).save_pretrained(OUT)   # WAJIB: tokenizer + chat_template ikut
print("merged fp16 ->", OUT)
del base, merged; gc.collect()

In [ ]:
# Sanity check: model merged fp16 masih jago (sebelum konversi)
m=AutoModelForCausalLM.from_pretrained(OUT, torch_dtype=torch.float16, device_map="auto")
t=AutoTokenizer.from_pretrained(OUT)
msgs=[{"role":"system","content":SYS},{"role":"user","content":"Pesan 3 kopi, harga 25.000 per item."}]
try: enc=t.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True, enable_thinking=False).to(m.device)
except TypeError: enc=t.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(m.device)
out=m.generate(**enc, max_new_tokens=64, do_sample=False, pad_token_id=t.eos_token_id)
print("merged fp16 output:", t.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True))
del m; gc.collect(); torch.cuda.empty_cache()

## 3. Konversi ke GGUF (llama.cpp)

**GGUF** = format tunggal yang dibaca Ollama & llama.cpp. `convert_hf_to_gguf.py` (perhatikan
*underscore*, bukan tanda hubung versi lama) sudah mengenali arsitektur `Qwen3ForCausalLM`.

In [ ]:
!git clone --depth 1 https://github.com/ggml-org/llama.cpp
!pip install -q -r llama.cpp/requirements.txt
!python llama.cpp/convert_hf_to_gguf.py ./qwen3-merged-fp16 --outtype bf16 --outfile qwen3-0.6b-spesialis-bf16.gguf
!ls -lh *.gguf

## 4. Quantize

Perkecil GGUF. Untuk spesialis mungil, **Q8_0** disarankan (~0.6 GB, near-lossless — hindari
kerusakan quant pada model yang baru fine-tuned). Q4_K_M kalau butuh lebih kecil.

In [ ]:
!cmake -S llama.cpp -B llama.cpp/build -DLLAMA_CURL=OFF >/dev/null 2>&1
!cmake --build llama.cpp/build --config Release -j --target llama-quantize 2>&1 | tail -2
Q="./llama.cpp/build/bin/llama-quantize"
!{Q} qwen3-0.6b-spesialis-bf16.gguf qwen3-0.6b-spesialis-Q8_0.gguf  Q8_0   2>&1 | tail -1
!{Q} qwen3-0.6b-spesialis-bf16.gguf qwen3-0.6b-spesialis-Q4_K_M.gguf Q4_K_M 2>&1 | tail -1
!ls -lh *.gguf

## 5. Push ke HuggingFace Hub

Jadikan HF Hub **jembatan distribusi**: push GGUF sekali, lalu device mana pun tinggal tarik.
Token `HF_TOKEN_WRITE` (WRITE) sudah ada di Colab Secrets; username `chmdznr`.

In [ ]:
from google.colab import userdata
from huggingface_hub import HfApi, create_repo

REPO="chmdznr/qwen3-0.6b-spesialis-gguf"
token=userdata.get("HF_TOKEN_WRITE")
create_repo(REPO, repo_type="model", exist_ok=True, token=token)
api=HfApi(token=token)
for f in ["qwen3-0.6b-spesialis-Q8_0.gguf","qwen3-0.6b-spesialis-Q4_K_M.gguf"]:
    api.upload_file(path_or_fileobj=f, path_in_repo=f, repo_id=REPO)
    print("pushed", f)
print("-> https://huggingface.co/"+REPO)

## 6. Demo HIDUP — Ollama di dalam Colab

Sebelum kirim ke device, jalankan di sini dulu. (Catatan: semua ini *ephemeral* — Colab hapus
saat disconnect. Untuk permanen, lihat PDF runbook.)

⚠️ `!ollama serve` **memblok sel selamanya** — kita jalankan di background pakai `subprocess`.

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, os, time, requests
subprocess.Popen(["ollama","serve"], stdout=open("/content/ollama.log","w"),
    stderr=subprocess.STDOUT, preexec_fn=os.setsid,
    env={**os.environ, "OLLAMA_HOST":"0.0.0.0:11434"})
for _ in range(30):
    try:
        if requests.get("http://localhost:11434/api/tags").ok: print("Ollama up"); break
    except Exception: time.sleep(1)

In [ ]:
# Tulis Modelfile (via list-baris agar tak bentrok triple-quote).
# PENTING: Ollama MENGABAIKAN chat-template Jinja di GGUF -> kita suplai TEMPLATE ChatML sendiri.
modelfile=[
 "FROM ./qwen3-0.6b-spesialis-Q8_0.gguf",
 'TEMPLATE """{{- if .System }}<|im_start|>system',
 "{{ .System }}<|im_end|>",
 "{{ end }}{{- range .Messages }}<|im_start|>{{ .Role }}",
 "{{ .Content }}<|im_end|>",
 "{{ end }}<|im_start|>assistant",
 '"""',
 'PARAMETER stop "<|im_start|>"',
 'PARAMETER stop "<|im_end|>"',
 "PARAMETER temperature 0.3",
 "SYSTEM Ekstrak pesanan ke JSON dengan key produk, jumlah, harga_satuan, total; angka tanpa titik; keluarkan HANYA JSON.",
]
open("Modelfile","w").write("\n".join(modelfile))
print(open("Modelfile").read())
print("---")
!ollama create qwen3-spesialis -f Modelfile

In [ ]:
import requests
from openai import OpenAI

client=OpenAI(base_url="http://localhost:11434/v1/", api_key="ollama")
r=client.chat.completions.create(model="qwen3-spesialis",
    messages=[{"role":"user","content":"Pesan 4 sabun, harga 8.500 per item."}])
print("output spesialis :", r.choices[0].message.content)

d=requests.post("http://localhost:11434/api/generate",
    json={"model":"qwen3-spesialis","prompt":"Pesan 2 beras harga 12.000","stream":False}).json()
print(f"throughput (T4)  : {d['eval_count']/d['eval_duration']*1e9:.1f} tok/dtk  (catatan: T4 != Jetson)")

## 7. Pola Client Universal (OpenAI API)

Inti deployment: **kode aplikasimu portabel.** Ollama lokal, vLLM cloud, NVIDIA NIM — semuanya
*OpenAI-compatible*. Pindah target = ganti **3 baris** (`base_url`, `api_key`, `model`).

In [ ]:
from openai import OpenAI
# Ganti hanya bagian ini untuk pindah target:
client = OpenAI(base_url="http://localhost:11434/v1/", api_key="ollama")   # 1) Ollama lokal (Mac/Jetson/Colab)
MODEL  = "qwen3-spesialis"
# 2) vLLM cloud :  OpenAI(base_url="http://<host>:8000/v1", api_key="EMPTY");  MODEL="Qwen/Qwen3-1.7B"
# 3) NVIDIA NIM :  OpenAI(base_url="https://integrate.api.nvidia.com/v1", api_key="nvapi-..."); MODEL="meta/llama-3.3-70b-instruct"
resp=client.chat.completions.create(model=MODEL,
    messages=[{"role":"user","content":"Pesan 5 kopi, harga 25.000 per item."}])
print(resp.choices[0].message.content)

## 8. Pohon Keputusan: Target → Tool

| Target | Tool | Kenapa |
|--------|------|--------|
| **Laptop / dev** | Ollama, llama.cpp (Metal) | termudah, lokal |
| **Edge / Jetson 8GB** | **Ollama** (mudah) → llama.cpp-CUDA (lebih dalam) | muat & cepat di GGUF Q4/Q8 |
| Edge produksi latency-kritis | TensorRT-LLM | *konsep* (build berat, butuh host NVIDIA) |
| **Cloud / banyak user** | vLLM, SGLang, NIM | throughput tinggi, GPU datacenter |
| (legacy) | TGI | maintenance mode sejak Des 2025 |

**Aturan jujur:** vLLM/TGI **bukan** untuk Orin Nano 8GB — itu tool datacenter. Untuk edge:
Ollama/llama.cpp/TensorRT. Tapi karena semua OpenAI-compatible, **kode tetap sama**.

## 9. Jembatan ke Device Asli + Ringkasan Modul 04

Notebook ini membuktikan pipeline & GGUF sudah di HF (`chmdznr/qwen3-0.6b-spesialis-gguf`).
Sekarang ikuti **PDF Runbook** untuk deploy di **Mac** & **Jetson**: cukup
`ollama run hf.co/chmdznr/qwen3-0.6b-spesialis-gguf` (tarik dari HF, nol scp) atau `ollama create`
dengan Modelfile, lalu benchmark dengan `tegrastats`.

### 🎉 Modul 04 lengkap — 8 notebook
00 Bangun · 01 Pakai · 02 Produksikan · 03 Adapt · 04 Ukur · 05 SLM spesialis · 06 Galeri ·
**07 Deploy ke edge**

**Selanjutnya: Module 05 — RAG.**